# 网站宣传册生成器

## 练习目标（理念）

用 **抓取网站 + OpenAI GPT-4o-mini** 自动生成公司宣传册（Markdown），并可翻译成多种语言。本笔记本在 Jupyter 里逐步演示：链接筛选 → 多页汇总 → 生成宣传册 → 流式输出 → 交互控件。

## 功能一览

- **网站分析**：自动抓取并清洗页面正文与链接
- **AI 生成**：用 `gpt-4o-mini` 判断相关链接、撰写宣传册
- **专业输出**：Markdown 格式，便于阅读与再编辑
- **多语言**：把宣传册翻译成目标语言
- **交互式**：单元格逐步跑，也可用 ipywidgets 点选
- **展示**：Jupyter 原生 Markdown / HTML 渲染

## 和本课的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `client.chat.completions.create(...)` |
| system / user messages | 链接筛选 prompt、宣传册 prompt、翻译 prompt |
| `response_format=json_object` | 让模型只返回 JSON 链接列表 |
| 流式 `stream=True` | `stream_brochure` / `stream_content` 边收边打印 |
| BeautifulSoup 抓取 | `Website` 类清洗 HTML |

## 怎么跑

1. Python 3.8+，Jupyter 环境；依赖：`openai python-dotenv requests beautifulsoup4 ipywidgets`
2. 在项目目录准备 `.env`：`OPENAI_API_KEY=your_api_key_here`（到 [OpenAI API keys](https://platform.openai.com/api-keys) 创建）
3. 从上到下运行；示例 URL 可改成你想分析的网站
4. 交互区可用小部件一键生成 / 翻译

```bash
pip install openai python-dotenv requests beautifulsoup4 ipywidgets
```


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv
# 导入标准库 os：读环境变量 OPENAI_API_KEY
import os
# 导入标准库 requests：HTTP GET 抓取网页
import requests
# 导入标准库 json：把模型返回的 JSON 字符串解析成 dict
import json
# 从 typing 导入 List：类型标注（本文件部分函数签名会用到）
from typing import List
# 从 bs4 导入 BeautifulSoup：解析 HTML，抽标题/正文/链接
from bs4 import BeautifulSoup
# 导入 ipywidgets：后面做交互式 URL / 语言控件
import ipywidgets as widgets
# 从 IPython.display 导入展示工具：Markdown / HTML / 清输出
from IPython.display import display, Markdown, HTML, clear_output
# 导入标准库 time：保存文件时写时间戳等
import time

# 运行时提示文案保留英文，便于对照原输出习惯
print("✅ All libraries imported successfully!")


## 配置

设置 OpenAI API 密钥并初始化客户端（Client）与抓取用的 HTTP 头。


In [ ]:
# ========== 配置：读 API Key、建 OpenAI 客户端、准备浏览器 User-Agent ==========

def get_client_and_headers():
    """Initialize OpenAI client and headers for web scraping"""
    # 加载 .env；override=True 用文件值覆盖已有环境变量
    load_dotenv(override=True)
    # 从环境读取密钥（不要把真实 key 写进笔记本）
    api_key = os.getenv("OPENAI_API_KEY")

    # 粗检密钥形态；打印文案保持英文原样
    if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
        print("✅ API key looks good!")
    else:
        print("⚠️  There might be a problem with your API key")
        print("Make sure you have set OPENAI_API_KEY in your .env file or environment variables")

    # 显式传入 api_key 创建客户端
    client = OpenAI(api_key=api_key)

    # 抓网页时伪装成常见 Chrome，降低被站点拦截的概率
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    return client, headers

# 初始化一次，供后续 Website / 生成函数复用
client, headers = get_client_and_headers()
print("✅ OpenAI client initialized successfully!")


## 核心功能

下面几格定义：展示工具、Website 抓取类、各类 prompt、宣传册生成与翻译。


In [ ]:
# ========== 展示工具：Markdown 一次显示 / 流式边收边打印 ==========

def display_content(content, is_markdown=True):
    """Display content using Jupyter's display methods"""
    # True → 按 Markdown 渲染；False → 普通 print
    if is_markdown:
        display(Markdown(content))
    else:
        print(content)

def stream_content(response, title="Content"):
    """
    Utility function to handle streaming content display in Jupyter

    Args:
        response: OpenAI streaming response object
        title (str): Title to display for the streaming content

    Returns:
        str: Complete streamed content
    """
    # 累积完整文本，最后返回给调用方
    result = ""

    # 用 HTML 显示彩色标题（title 参数原样嵌入）
    display(HTML(f"<h3 style='color: #1f77b4;'>{title}...</h3>"))

    # 与顶部 import 重复：保持原逻辑（局部再导入一次）
    from IPython.display import clear_output
    import time

    # 遍历流式 chunk：取出 delta.content（可能为 None）
    for chunk in response:
        content = chunk.choices[0].delta.content or ""
        result += content
        # end='' 不换行；flush=True 立刻刷到输出，形成「打字机」效果
        print(content, end='', flush=True)

    # 完成分隔线与 COMPLETE 提示（英文文案保留）
    display(HTML(f"<div style='color: green; font-weight: bold; margin-top: 20px;'>{'='*50}</div>"))
    display(HTML(f"<div style='color: green; font-weight: bold;'>{title.upper()} COMPLETE</div>"))
    display(HTML(f"<div style='color: green; font-weight: bold;'>{'='*50}</div>"))

    return result

print("✅ Utility functions loaded!")


In [ ]:
# ========== Website 类：抓取单个 URL → 标题 / 正文 / 链接列表 ==========

class Website:
    def __init__(self, url):
        # 保存地址
        self.url = url
        # 每次实例化都重新取 client/headers（与原逻辑一致）
        self.client, self.headers = get_client_and_headers()
        print(f"🌐 Fetching content from: {url}")
        # GET 下载页面
        response = requests.get(url, headers=self.headers)
        # 原始字节交给 BeautifulSoup
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        # <title> 缺失时用英文占位（影响后续 prompt 格式，勿改）
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            # 去掉对宣传册无用的 script/style/img/input
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            # 抽出可见文本
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        # 收集所有 href，过滤空值
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]
        print(f"✅ Website analyzed: {self.title}")

    def get_contents(self):
        # 拼成「标题 + 正文」字符串，供宣传册 user prompt 使用（英文标签保留）
        return f"Webpage Title: {self.title}\nWebpage Contents: {self.text}\n\n"

print("✅ Website class loaded!")


In [ ]:
# ========== AI prompt 工厂：链接筛选 / 宣传册 / 翻译（字符串内容勿译）==========

def get_links_system_prompt():
    # system：教模型从链接列表里挑「适合写公司宣传册」的页面
    link_system_prompt = """"You are provided with a list of links found on a webpage. \
        You are able to decide which of the links would be most relevant to include in a brochure about the company. \
        Relevant links usually include: About page, or a Company page, or Careers/Jobs pages or News page\n"""
    link_system_prompt += "Always respond in JSON exactly like this: \n"
    link_system_prompt += """
        {
            "links": [
                {"type": "<page type>", "url": "<full URL>"},
                {"type": "<page type>", "url": "<full URL>"}
            ]
        }\n
    """
    link_system_prompt += """ If no relevant links are found, return:
        {
            "links": []
        }\n
    """
    link_system_prompt += "If multiple links could map to the same type (e.g. two About pages), include the best candidate only.\n"

    link_system_prompt += "You should respond in JSON as in the below examples:\n"
    # 下面「实施例」是原笔记本中文小标题，夹在英文 few-shot 里；prompt 正文保持原样
    link_system_prompt += """
        # 实施例1
        Input links:
        - https://acme.com/about  
        - https://acme.com/pricing  
        - https://acme.com/blog  
        - https://acme.com/signup  

        Output:
        {
        "links": [
            {"type": "about page", "url": "https://acme.com/about"},
            {"type": "blog page", "url": "https://acme.com/blog"},
            {"type": "pricing page", "url": "https://acme.com/pricing"}
        ]
        }
        """
    link_system_prompt += """
        # 实施例2
        Input links:
        - https://startup.io/  
        - https://startup.io/company  
        - https://startup.io/careers  
        - https://startup.io/support  

        Output:
        {
        "links": [
            {"type": "company page", "url": "https://startup.io/company"},
            {"type": "careers page", "url": "https://startup.io/careers"}
        ]
        }
        """
    link_system_prompt += """
        # 实施例3
        Input links:
        - https://coolapp.xyz/login  
        - https://coolapp.xyz/random  

        Output:
        {
        "links": []
        }
        """
    return link_system_prompt

def get_links_user_prompt(website):
    # user：把该站全部 links 塞给模型，要求返回完整 https JSON
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \n"
    user_prompt += "Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

def get_brochure_system_prompt():
    # system：写给潜在客户/投资人/求职者看的短宣传册，Markdown 输出
    brochure_system_prompt = """
        You are an assistant that analyzes the contents of several relevant pages from a company website \
        and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.
        Include details of company culture, customers and careers/jobs if you have the information.
    """
    return brochure_system_prompt

def get_brochure_user_prompt(url):
    # user：附上落地页 + 相关页正文；截断到 15000 字符控制上下文长度
    user_prompt = f"You are looking at a company details of: {url}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_details_for_brochure(url)
    user_prompt = user_prompt[:15000] # Truncate if more than 15,000 characters
    return user_prompt

def get_translation_system_prompt(target_language):
    # system：专业商务翻译角色；目标语言由参数注入
    translation_system_prompt = f"You are a professional translator specializing in business and marketing content. \
    Translate the provided brochure to {target_language} while maintaining all formatting and professional tone."
    return translation_system_prompt

def get_translation_user_prompt(original_brochure, target_language):
    # user：附带翻译准则 + 原文宣传册（准则英文保留）
    translation_prompt = f"""
    You are a professional translator. Please translate the following brochure content to {target_language}.
    
    Important guidelines:
    - Maintain the markdown formatting exactly as it appears
    - Keep all headers, bullet points, and structure intact
    - Translate the content naturally and professionally
    - Preserve any company names, product names, or proper nouns unless they have established translations
    - Maintain the professional tone and marketing style
    
    Brochure content to translate:
    {original_brochure}
    """
    return translation_prompt

print("✅ AI prompt functions loaded!")


In [ ]:
# ========== 核心链路：筛链接 → 拼多页详情 → 生成 / 流式宣传册 ==========

def get_links(url):
    """Get relevant links from a website using AI analysis"""
    # 先抓首页，拿到 links 列表
    website = Website(url)
    # json_object：强制模型返回可解析 JSON
    response = website.client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": get_links_system_prompt()},
            {"role": "user", "content": get_links_user_prompt(website)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    print("🔗 Found relevant links:", result)
    # 字符串 → Python dict
    return json.loads(result)

def get_details_for_brochure(url):
    """Get comprehensive details from website and relevant pages"""
    website = Website(url)
    # 先放落地页
    result = "Landing page:\n"
    result += website.get_contents()
    # AI 选出相关内链
    links = get_links(url)
    print("📄 Analyzing additional pages...")
    # 逐个抓取相关页，按 type 标题拼进大文本
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

def create_brochure(url):
    """Create a brochure from a website URL"""
    website = Website(url)
    print("🤖 Generating brochure with AI...")
    # 非流式：等整段生成完再 Markdown 展示
    response = website.client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": get_brochure_system_prompt()},
            {"role": "user", "content": get_brochure_user_prompt(url)}
        ]
    )
    result = response.choices[0].message.content
    display_content(result, is_markdown=True)
    return result

def stream_brochure(url):
    """Create a brochure with streaming output"""
    website = Website(url)
    print("🤖 Generating brochure with streaming output...")
    # stream=True：边生成边交给 stream_content 打印
    response = website.client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": get_brochure_system_prompt()},
            {"role": "user", "content": get_brochure_user_prompt(url)}
        ],
        stream=True
    )

    # 使用可重用的流实用程序函数
    result = stream_content(response, "Generating brochure")
    return result

print("✅ Core brochure generation functions loaded!")


In [ ]:
# ========== 翻译：先生成英文宣传册，再译成目标语言（可流式）==========

def translate_brochure(url, target_language="Spanish", stream_mode=False):
    """
    Generate a brochure and translate it to the target language

    Args:
        url (str): The website URL to generate brochure from
        target_language (str): The target language for translation (default: "Spanish")
        stream_mode (bool): Whether to use streaming output (default: False)

    Returns:
        str: Translated brochure content
    """
    # 先走 create_brochure 拿到原文
    print(f"🌍 Generating brochure and translating to {target_language}...")
    original_brochure = create_brochure(url)

    # 组装翻译用的 system / user
    translation_system_prompt = get_translation_system_prompt(target_language)
    translation_user_prompt = get_translation_user_prompt(original_brochure, target_language)

    # 再拿一个 Website 实例主要是为了复用其上的 client
    website = Website(url)

    if stream_mode:
        # 流式翻译路径
        response = website.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": translation_system_prompt},
                {"role": "user", "content": translation_user_prompt}
            ],
            stream=True
        )

        translated_brochure = stream_content(response, f"Translating brochure to {target_language}")
    else:
        # 非流式：整段返回后再 Markdown 展示
        response = website.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": translation_system_prompt},
                {"role": "user", "content": translation_user_prompt}
            ]
        )

        translated_brochure = response.choices[0].message.content

        display_content(translated_brochure, is_markdown=True)

    return translated_brochure

print("✅ Translation functions loaded!")


## 互动示例

下面用几个公开网站试跑宣传册生成。你可以改 URL 后重新运行对应单元格。


In [ ]:
# ========== 示例 1：非流式生成宣传册 ==========
# 把 sample_url 改成你想分析的任何网站即可

sample_url = "https://openai.com"  # Change this to any website you want to analyze

print(f"🚀 Generating brochure for: {sample_url}")
print("=" * 60)

# 调用 create_brochure：抓站 → 筛链 → 写 Markdown 宣传册
brochure = create_brochure(sample_url)


In [ ]:
# ========== 示例 2：流式生成宣传册（边生成边打印）==========

streaming_url = "https://anthropic.com"  # Change this to any website you want to analyze

print(f"🚀 Generating brochure with streaming for: {streaming_url}")
print("=" * 60)

# stream_brochure 内部 stream=True + stream_content
streaming_brochure = stream_brochure(streaming_url)


In [ ]:
# ========== 示例 3：生成后翻译到指定语言 ==========

translation_url = "https://huggingface.co"  # Change this to any website you want to analyze
target_language = "Spanish"  # Change this to any language you want

print(f"🚀 Generating and translating brochure for: {translation_url}")
print(f"🌍 Target language: {target_language}")
print("=" * 60)

# stream_mode=False：等整段翻译完再展示
translated_brochure = translate_brochure(translation_url, target_language, stream_mode=False)


## 交互式小部件界面

用下面的控件输入 URL、选语言、勾选是否流式 / 是否翻译，再点按钮生成。


In [ ]:
# ========== ipywidgets 交互界面：URL + 语言 + 流式/翻译开关 ==========

# 再导入一次（与顶部重复，保持原逻辑）
import ipywidgets as widgets
from IPython.display import display, clear_output

# URL 输入框：默认 openai.com
url_input = widgets.Text(
    value='https://openai.com',
    placeholder='Enter website URL (e.g., https://example.com)',
    description='Website URL:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# 目标语言下拉（翻译时用）
language_dropdown = widgets.Dropdown(
    options=['English', 'Spanish', 'French', 'German', 'Chinese', 'Japanese', 'Portuguese', 'Italian'],
    value='English',
    description='Language:',
    style={'description_width': 'initial'}
)

# 是否流式输出
stream_checkbox = widgets.Checkbox(
    value=False,
    description='Use streaming output',
    style={'description_width': 'initial'}
)

# 是否先生成再翻译
translate_checkbox = widgets.Checkbox(
    value=False,
    description='Translate brochure',
    style={'description_width': 'initial'}
)

# 触发按钮
generate_button = widgets.Button(
    description='Generate Brochure',
    button_style='success',
    icon='rocket'
)

# 结果输出区域（点击后在此 clear + 打印）
output_area = widgets.Output()

def on_generate_clicked(b):
    # 所有打印都进 output_area，避免污染其它单元格输出
    with output_area:
        clear_output(wait=True)
        url = url_input.value.strip()

        # 空 URL 直接提示（英文文案保留）
        if not url:
            print("❌ Please enter a valid URL")
            return

        # 没写协议时默认补 https://
        if not url.startswith(('http://', 'https://')):
            url = 'https://' + url

        print(f"🚀 Generating brochure for: {url}")
        print("=" * 60)

        try:
            if translate_checkbox.value:
                # 生成并翻译
                result = translate_brochure(url, language_dropdown.value, stream_mode=stream_checkbox.value)
            else:
                # 仅生成
                if stream_checkbox.value:
                    result = stream_brochure(url)
                else:
                    result = create_brochure(url)

            print("\n✅ Brochure generation completed!")

        except Exception as e:
            # 错误提示保留英文，便于对照排错笔记本
            print(f"❌ Error generating brochure: {str(e)}")
            print("Please check your API key and internet connection.")

# 绑定点击回调
generate_button.on_click(on_generate_clicked)

# 画出控件 UI
print("🎯 Interactive Brochure Generator")
print("Enter a website URL and click 'Generate Brochure' to create a professional brochure!")
print()

display(url_input)
display(widgets.HBox([language_dropdown, stream_checkbox, translate_checkbox]))
display(generate_button)
display(output_area)


## 高级使用示例

批量分析多个网站，或一次生成多种语言版本。


In [ ]:
# ========== 高级示例 1：批量分析多个网站并收集结果 ==========

websites_to_analyze = [
    "https://openai.com",
    "https://anthropic.com",
    "https://huggingface.co"
]

print("🔍 Analyzing multiple websites...")
print("=" * 60)

# url → 宣传册正文
brochures = {}
for url in websites_to_analyze:
    print(f"\n📊 Generating brochure for: {url}")
    try:
        brochure = create_brochure(url)
        brochures[url] = brochure
        print(f"✅ Successfully generated brochure for {url}")
    except Exception as e:
        print(f"❌ Failed to generate brochure for {url}: {str(e)}")

    print("-" * 40)

print(f"\n🎉 Generated {len(brochures)} brochures successfully!")


In [ ]:
# ========== 高级示例 2：同一网站生成多种语言宣传册 ==========

target_website = "https://openai.com"  # Change this to any website
languages = ["Spanish", "French", "German", "Chinese"]

print(f"🌍 Generating brochures in multiple languages for: {target_website}")
print("=" * 60)

# language → 译文
multilingual_brochures = {}
for language in languages:
    print(f"\n🔄 Translating to {language}...")
    try:
        translated_brochure = translate_brochure(target_website, language, stream_mode=False)
        multilingual_brochures[language] = translated_brochure
        print(f"✅ Successfully translated to {language}")
    except Exception as e:
        print(f"❌ Failed to translate to {language}: {str(e)}")

    print("-" * 40)

print(f"\n🎉 Generated brochures in {len(multilingual_brochures)} languages!")


## 自定义函数

针对「存文件 / 聚焦主题 / 快速分析」三类场景的扩展函数。


In [ ]:
# ========== 自定义扩展：存文件 / 带焦点的宣传册 / 快速网站分析 ==========

def save_brochure_to_file(brochure_content, filename, url):
    """Save brochure content to a markdown file"""
    try:
        # utf-8 写入 Markdown 文件
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"# Brochure for {url}\n\n")
            f.write(f"Generated on: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            f.write("---\n\n")
            f.write(brochure_content)
        print(f"✅ Brochure saved to: {filename}")
        return True
    except Exception as e:
        print(f"❌ Error saving brochure: {str(e)}")
        return False

def generate_custom_brochure(url, focus_areas=None):
    """Generate a brochure with focus on specific areas"""
    # 默认关注：总览 / 产品 / 文化 / 招聘
    if focus_areas is None:
        focus_areas = ["company overview", "products", "culture", "careers"]

    website = Website(url)

    # 自定义 system：把焦点列表写进指令（英文句子结构保留）
    custom_system_prompt = f"""
    You are an assistant that analyzes website content and creates a professional brochure.
    Focus specifically on these areas: {', '.join(focus_areas)}.
    Create a markdown brochure that emphasizes these aspects for prospective customers, investors and recruits.
    """

    response = website.client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": custom_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(url)}
        ]
    )

    result = response.choices[0].message.content
    display_content(result, is_markdown=True)
    return result

def quick_website_analysis(url):
    """Perform a quick analysis of a website without generating full brochure"""
    website = Website(url)

    # 注意：原字符串里中英混排标题保留；链接区英文注释也保留在 f-string 内
    analysis = f"""
    # 快速网站分析：{url}
    
    **Title:** {website.title}
    **Total Links Found:** {len(website.links)}
    **Content Length:** {len(website.text)} characters
    
    # 示例内容（前 500 个字符）：
    {website.text[:500]}...
    
    # 所有链接：
    {chr(10).join(website.links[:10])}  # Show first 10 links
    """

    display_content(analysis, is_markdown=True)
    return analysis

print("✅ Custom functions loaded!")


## 自定义函数的使用示例

试跑快速分析、带焦点宣传册、以及保存到本地 Markdown 文件。


In [ ]:
# ========== 示例：快速网站分析（不生成完整宣传册）==========

test_url = "https://openai.com"  # Change this to any website

print("🔍 Performing quick website analysis...")
print("=" * 50)

quick_analysis = quick_website_analysis(test_url)


In [ ]:
# ========== 示例：按指定焦点领域生成自定义宣传册 ==========

custom_url = "https://anthropic.com"  # Change this to any website
focus_areas = ["AI safety", "research", "products", "team"]  # Custom focus areas

print("🎯 Generating custom brochure with specific focus...")
print(f"Focus areas: {', '.join(focus_areas)}")
print("=" * 50)

custom_brochure = generate_custom_brochure(custom_url, focus_areas)


In [ ]:
# ========== 示例：生成宣传册并保存为 .md 文件 ==========

save_url = "https://huggingface.co"  # Change this to any website

print("💾 Generating brochure and saving to file...")
print("=" * 50)

# 先生成
brochure_content = create_brochure(save_url)

# 用 URL 拼一个简单文件名（替换 https:// 与 /）
filename = f"brochure_{save_url.replace('https://', '').replace('/', '_')}.md"
save_success = save_brochure_to_file(brochure_content, filename, save_url)

if save_success:
    print(f"📁 You can find the saved brochure in: {filename}")
else:
    print("❌ Failed to save brochure to file")


## 故障排除和提示

### 常见问题与解决办法

1. **API 密钥问题**
   - 确认 `.env` 里已设置 `OPENAI_API_KEY`
   - 确认账户有足够额度
   - 检查密钥是否以 `sk-proj-` 开头（本笔记本的粗检逻辑如此）

2. **网站抓取问题**
   - 部分站点会拦截自动化请求
   - 一个站点失败时换另一个试试
   - 工具已带标准 `User-Agent`，可避开最基础的拦截

3. **内存 / 上下文过长**
   - 大站点文本很长；工具会把 user prompt 截断到 15,000 字符

4. **速率限制（Rate Limit）**
   - OpenAI 有调用频率限制；触顶后等几分钟再试

### 获得更好结果的技巧

1. **选结构清晰的网站**：有 About / Products / Careers 的站点效果最好；少用「几乎全是图或强依赖 JS」的站
2. **长文用流式**：勾选 streaming，边生成边看进度
3. **自定义焦点**：用 `generate_custom_brochure` 聚焦特定主题
4. **保存成果**：用 `save_brochure_to_file` 落盘 Markdown，方便后续编辑


## 结语

这个 Jupyter 笔记本把「网站 → 宣传册」做成可逐步运行的完整界面。你可以：

- 从任意网站生成专业 Markdown 宣传册
- 翻译成多种语言
- 用交互小部件快速操作
- 把结果保存成文件
- 做快速网站分析
- 按焦点领域定制宣传册
- 用流式输出获得实时反馈

### 建议的下一步

1. 先用上面的小部件对你喜欢的网站试一发
2. 换不同类型的 URL 做对比
3. 试翻译功能
4. 用保存函数归档生成结果
5. 用自定义焦点写更有针对性的宣传册

### 支持

出问题时：先看上一格故障排除 → 核对 API Key → 检查网络 → 换一个网站再试。

祝你玩得开心！
